# Step 7: volatility

- **Target:** how big day t's move is: the log of log(High / Low). A number, not up/down.
- **Models:**
  - **HAR:** linear regression on the average range over the last 1, 5 and 22 days. The standard simple model.
  - **Random forest** on the same three inputs.
  - **GRU** on the last 20 daily ranges, with three random seeds.
- **R2:** 1 is perfect, and 0 is no better than guessing the average of all earlier days.
- Same six walk-forward folds as before.

In [1]:
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

from data import load_all, load_us
from evaluation import paired_loss_test
from features import log_range, make_har, make_har_with_us, make_range_sequence
from gru import HIDDEN_SIZES
from volatility import (fit_gru_vol, fit_regressor, make_vol_dataset, pick_gru_vol_setting, qlike,
                        vol_scores, walk_forward_garch, walk_forward_vol)

markets = load_all()

## Is there structure to learn?

How much today resembles the recent past (correlation with 1, 5 and 22 days earlier). Direction has almost none; the range has plenty.

In [2]:
rows = {}
for market, df in markets.items():
    up = (df['ret'] > 0).astype(float)
    size = log_range(df)
    rows[market] = {
        "direction, 1 day": up.autocorr(1),
        "range, 1 day": size.autocorr(1),
        "range, 5 days": size.autocorr(5),
        "range, 22 days": size.autocorr(22),
    }
pd.DataFrame(rows).T.round(2)

,"direction, 1 day","range, 1 day","range, 5 days","range, 22 days"
Indonesia,-0.01,0.51,0.33,0.22
Malaysia,0.03,0.48,0.38,0.28
Philippines,0.02,0.40,0.26,0.13
Singapore,-0.01,0.45,0.34,0.22
Thailand,0.02,0.54,0.39,0.30
India,0.08,0.44,0.34,0.21


## Run the models

Three markets run at a time. About 20 minutes, mostly the GRU.

Each finished run is saved in `results/`, so a rerun skips it. **Delete that folder after changing a model.**

In [3]:
RESULTS = Path('results')
RESULTS.mkdir(exist_ok=True)

def run_market(path, run, df):
    """Load the saved predictions, or run the model and save them."""
    if path.exists():
        return pd.read_pickle(path)
    preds = run(df)
    preds.to_pickle(path)
    return preds

def run_all_markets(name, run):
    runs = Parallel(n_jobs=3)(
        delayed(run_market)(RESULTS / f'{name}_{market}.pkl', run, df)
        for market, df in markets.items()
    )
    return dict(zip(markets, runs))

def sklearn_run(settings, make_x):
    return partial(walk_forward_vol, fit=fit_regressor, settings=settings, make_x=make_x)

def gru_run(seed):
    return partial(walk_forward_vol, fit=partial(fit_gru_vol, seed=seed), settings=HIDDEN_SIZES,
                   make_x=make_range_sequence, pick=partial(pick_gru_vol_setting, seed=seed))

LINE = [LinearRegression()]
FOREST = [RandomForestRegressor(n_estimators=300, min_samples_leaf=leaf, random_state=0) for leaf in [10, 50, 200]]
make_har_us = partial(make_har_with_us, us=load_us())

har = run_all_markets('har', sklearn_run(LINE, make_har))
har_us = run_all_markets('har_us', sklearn_run(LINE, make_har_us))
garch = run_all_markets('garch', walk_forward_garch)
forest = run_all_markets('forest', sklearn_run(FOREST, make_har))
gru = {seed: run_all_markets(f'gru_seed{seed}', gru_run(seed)) for seed in [0, 1, 2]}

## All six test years pooled

**p vs HAR** is a paired test of squared errors on the same days (Diebold-Mariano).

In [4]:
def r2_of(guess, preds):
    """R2 of any guess, against the average of all earlier days."""
    error = guess - preds["actual"]
    error_of_average = preds["average_so_far"] - preds["actual"]
    return 1 - (error ** 2).sum() / (error_of_average ** 2).sum()

rows = {}
for market in markets:
    h, f, g = har[market], forest[market], gru[0][market]
    baseline = h["predicted"]
    rows[market] = {
        "same as yesterday R2": r2_of(h["same_as_yesterday"], h),
        "HAR R2": r2_of(h["predicted"], h),
        "forest R2": r2_of(f["predicted"], f),
        "GRU R2": r2_of(g["predicted"], g),
        "p forest vs HAR": vol_scores(f, baseline)["p vs baseline"],
        "p GRU vs HAR": vol_scores(g, baseline)["p vs baseline"],
        "p HAR vs same as yesterday": vol_scores(h, h["same_as_yesterday"])["p vs baseline"],
    }
pd.DataFrame(rows).T.round(3)

,same as yesterday R2,HAR R2,forest R2,GRU R2,p forest vs HAR,p GRU vs HAR,p HAR vs same as yesterday
Indonesia,0.167,0.412,0.356,0.395,0.006,0.008,0.0
Malaysia,0.136,0.430,0.389,0.354,0.003,0.000,0.0
Philippines,-0.099,0.274,0.191,0.233,0.001,0.014,0.0
Singapore,-0.061,0.307,0.199,0.169,0.007,0.001,0.0
Thailand,0.242,0.486,0.395,0.443,0.000,0.000,0.0
India,0.137,0.408,0.311,0.393,0.001,0.103,0.0


## Do the GRU's results depend on the seed?

In [5]:
rows = {}
for market in markets:
    for seed in gru:
        g = gru[seed][market]
        rows[market, seed] = {
            "GRU R2": r2_of(g["predicted"], g),
            "p vs HAR": vol_scores(g, har[market]["predicted"])["p vs baseline"],
        }
pd.DataFrame(rows).T.round(3).unstack()

GRU R2               p vs HAR              
                 0      1      2        0      1      2
India        0.393  0.354  0.389    0.103  0.008  0.038
Indonesia    0.395  0.400  0.401    0.008  0.169  0.127
Malaysia     0.354  0.356  0.339    0.000  0.000  0.000
Philippines  0.233  0.247  0.256    0.014  0.009  0.012
Singapore    0.169  0.238  0.244    0.001  0.007  0.003
Thailand     0.443  0.419  0.435    0.000  0.001  0.000

## Year by year

HAR's R2 in each test year, and how many of the 36 market-years each model beats HAR.

In [6]:
def yearly_r2(preds_by_market):
    return pd.DataFrame({
        market: {year: r2_of(f["predicted"], f) for year, f in p.groupby("year")}
        for market, p in preds_by_market.items()
    }).T

har_years = yearly_r2(har)
display(har_years.round(2))
pd.Series({
    "forest": (yearly_r2(forest) > har_years).sum().sum(),
    "GRU, seed 0": (yearly_r2(gru[0]) > har_years).sum().sum(),
    "GRU, seed 1": (yearly_r2(gru[1]) > har_years).sum().sum(),
    "GRU, seed 2": (yearly_r2(gru[2]) > har_years).sum().sum(),
}, name="market-years above HAR, out of 36").to_frame()

,2017,2018,2019,2020,2021,2022
Indonesia,0.46,0.21,0.20,0.63,0.32,0.24
Malaysia,0.39,0.25,0.03,0.69,0.32,0.32
Philippines,0.15,0.27,0.13,0.51,0.16,0.15
Singapore,0.12,0.16,0.07,0.62,0.05,0.10
Thailand,0.70,0.06,0.24,0.68,0.19,0.20
India,0.53,0.27,0.12,0.61,0.25,0.23


,"market-years above HAR, out of 36"
forest,1
"GRU, seed 0",12
"GRU, seed 1",14
"GRU, seed 2",11


## Two more models: GARCH and HAR + US move

- **GARCH(1,1):** the classic volatility model. It forecasts the size of returns, so a straight line fitted on the years before the test year turns that into a range forecast.
- **HAR + US:** HAR plus the log size of the last US session's move, known before Asia opens.

**US weight** is HAR + US's weight on the US move, fitted on 2012-2022 (both sides are logs).

In [7]:
rows = {}
for market, df in markets.items():
    h, g, u = har[market], garch[market], har_us[market]
    X, y = make_vol_dataset(df, make_har_us)
    keep = X.index.year <= 2022
    rows[market] = {
        "HAR R2": r2_of(h["predicted"], h),
        "GARCH R2": r2_of(g["predicted"], g),
        "HAR + US R2": r2_of(u["predicted"], u),
        "p GARCH vs HAR": vol_scores(g, h["predicted"])["p vs baseline"],
        "p HAR + US vs HAR": vol_scores(u, h["predicted"])["p vs baseline"],
        "US weight": LinearRegression().fit(X[keep], y[keep]).coef_[-1],
    }
pd.DataFrame(rows).T.round(3)

,HAR R2,GARCH R2,HAR + US R2,p GARCH vs HAR,p HAR + US vs HAR,US weight
Indonesia,0.412,0.314,0.420,0.000,0.018,0.046
Malaysia,0.430,0.386,0.434,0.004,0.004,0.030
Philippines,0.274,0.237,0.280,0.006,0.362,0.061
Singapore,0.307,0.294,0.325,0.528,0.001,0.053
Thailand,0.486,0.448,0.488,0.004,0.071,0.019
India,0.408,0.365,0.408,0.005,0.723,0.013


## Scores 2020 can't dominate

R2 here is already on the log of the range, so a wild day doesn't get extra weight for being wild. 2020 scores high for another reason: the average of earlier days is a bad guess that year. Typical errors, all six markets together:

In [8]:
all_har = pd.concat(har.values())
pd.DataFrame({
    "HAR": (all_har["predicted"] - all_har["actual"]) ** 2,
    "average so far": (all_har["average_so_far"] - all_har["actual"]) ** 2,
}).groupby(all_har["year"]).mean().pow(0.5).T.round(3)

year,2017,2018,2019,2020,2021,2022
HAR,0.393,0.415,0.391,0.462,0.411,0.396
average so far,0.530,0.468,0.420,0.757,0.467,0.446


Three checks that don't rely on 2020:

- **R2 without 2020.**
- **QLIKE:** how far off the forecast is in relative terms, like "2x too high". Lower is better. Shown with and without 2020.
- **p vs HAR:** the paired test on daily QLIKE, all six years.

In [9]:
MODELS = {'HAR': har, 'HAR + US': har_us, 'GARCH': garch, 'forest': forest,
          'GRU 0': gru[0], 'GRU 1': gru[1], 'GRU 2': gru[2]}

def without_2020(p):
    return p[p["year"] != 2020]

def table(score):
    return pd.DataFrame({name: {m: score(runs[m], m) for m in markets} for name, runs in MODELS.items()}).round(3)

print("R2 without 2020")
display(table(lambda p, m: r2_of(without_2020(p)['predicted'], without_2020(p))))
print("QLIKE, all six years")
display(table(lambda p, m: qlike(p).mean()))
print("QLIKE without 2020")
display(table(lambda p, m: qlike(without_2020(p)).mean()))
print("p vs HAR on QLIKE, all six years")
table(lambda p, m: paired_loss_test(qlike(p), qlike(har[m])) if p is not har[m] else float('nan'))

R2 without 2020


,HAR,HAR + US,GARCH,forest,GRU 0,GRU 1,GRU 2
Indonesia,0.299,0.303,0.175,0.261,0.273,0.293,0.289
Malaysia,0.278,0.282,0.232,0.243,0.268,0.277,0.265
Philippines,0.169,0.172,0.108,0.131,0.161,0.159,0.161
Singapore,0.098,0.115,0.091,0.067,0.059,0.081,0.077
Thailand,0.387,0.388,0.333,0.295,0.338,0.340,0.325
India,0.311,0.311,0.249,0.252,0.301,0.289,0.304


QLIKE, all six years


,HAR,HAR + US,GARCH,forest,GRU 0,GRU 1,GRU 2
Indonesia,0.477,0.463,0.644,0.551,0.489,0.501,0.499
Malaysia,0.575,0.568,0.615,0.650,0.742,0.741,0.771
Philippines,0.629,0.609,0.673,0.865,0.761,0.707,0.678
Singapore,0.378,0.354,0.362,0.521,0.560,0.460,0.452
Thailand,0.366,0.360,0.377,0.458,0.390,0.463,0.392
India,0.507,0.504,0.491,0.655,0.472,0.583,0.497


QLIKE without 2020


,HAR,HAR + US,GARCH,forest,GRU 0,GRU 1,GRU 2
Indonesia,0.410,0.405,0.572,0.415,0.419,0.420,0.424
Malaysia,0.528,0.524,0.572,0.554,0.545,0.531,0.530
Philippines,0.542,0.548,0.633,0.572,0.553,0.560,0.561
Singapore,0.337,0.321,0.333,0.352,0.365,0.344,0.348
Thailand,0.309,0.305,0.338,0.330,0.311,0.313,0.314
India,0.405,0.404,0.444,0.428,0.395,0.403,0.391


p vs HAR on QLIKE, all six years


,HAR,HAR + US,GARCH,forest,GRU 0,GRU 1,GRU 2
Indonesia,NaN,0.003,0.000,0.092,0.172,0.088,0.033
Malaysia,NaN,0.011,0.086,0.045,0.001,0.002,0.007
Philippines,NaN,0.277,0.200,0.075,0.075,0.042,0.006
Singapore,NaN,0.000,0.317,0.027,0.005,0.016,0.006
Thailand,NaN,0.005,0.562,0.090,0.150,0.143,0.094
India,NaN,0.054,0.734,0.040,0.299,0.062,0.670


## What is the model paying attention to?

HAR's weights, fitted on 2012-2022. Because it's linear you can read them directly: a larger weight means that period counts more.

In [10]:
weights = {}
for market, df in markets.items():
    X, y = make_vol_dataset(df, make_har)
    keep = X.index.year <= 2022
    model = LinearRegression().fit(X[keep], y[keep])
    weights[market] = dict(zip(X.columns, model.coef_))
pd.DataFrame(weights).T.round(2)

,range_1d,range_5d,range_22d
Indonesia,0.27,0.29,0.29
Malaysia,0.16,0.37,0.34
Philippines,0.20,0.31,0.27
Singapore,0.14,0.35,0.35
Thailand,0.23,0.35,0.30
India,0.13,0.40,0.31


## Data check: Indonesia's fake opens

In 2020, Indonesia's fake-open days have bigger ranges, so those Highs and Lows may be affected. HAR's R2 with and without those days, and with and without 2020:

In [11]:
p = har["Indonesia"]
df = markets["Indonesia"]
fake_open = (df["Open"] == df["Close"].shift(1))[p.index]
in_2020 = p["year"] == 2020
groups = {
    "all test days": p,
    "real opens": p[~fake_open],
    "not 2020, all days": p[~in_2020],
    "not 2020, real opens": p[~in_2020 & ~fake_open],
    "2020, real opens": p[in_2020 & ~fake_open],
    "2020, fake opens": p[in_2020 & fake_open],
}
display(pd.DataFrame({name: {"days": len(g), "HAR R2": r2_of(g["predicted"], g)} for name, g in groups.items()}).T.round(3))
fake_open.groupby(p["year"]).sum().rename("fake-open days").to_frame()

,days,HAR R2
all test days,1452.0,0.412
real opens,1280.0,0.320
"not 2020, all days",1215.0,0.299
"not 2020, real opens",1151.0,0.303
"2020, real opens",129.0,0.418
"2020, fake opens",108.0,0.734


,fake-open days
year,
2017,1
2018,0
2019,0
2020,108
2021,0
2022,63
